# Catalogs
## Catalog class
`PTO` has a catalog system available to the user. The main implementation currently is for the NASA exoplanet archive. However, the framework is build generically enough that it can be expanded upon.

PTO has a implemented class `CatalogComposite` and `CatalogFull` in the `PTO.database.catalog` module that provides the general framework for all the databases. The core difference between these is based on number of entries for single target, following the style of NASA exoplanet archive. That is: `CatalogComposite` includes a single entry for each target, `CatalogFull` includes multiple entries for each target with separate references. 

Creating a catalog is very simple:

In [1]:
import PTO.database.catalog as cat

composite = cat.CatalogComposite()
full = cat.CatalogFull()

Unfortunately, those catalogs are empty. We could fill them manually, by playing with the content of the instances. However, most use cases involve a full sample of exoplanet population, or custom set of ephemeris. There are easier way to implement these.

Before we go further though, lets quickly describe the attributes of the catalogs:

 - `composite.table` and `full.table`
    - Table of the sample. This can be filtered down by criteria and is used by the transit window planner.
 - `composite.legacy_table` and `full.legacy_table`
    - Original table of the sample. This is saved after loading of the table. It is used for plotting purposes, in cases where the full exoplanet population is necessary.
 - `composite.filename` and `full.filename`
    - Filename where the table is saved. This is used for quick rerunning, as some TAP calls to the NASA archive take long time.
 - `composite.drop_mode` and `full.drop_mode`
    - Whether values with no errorbars are dropped or replaced with 0. If our value of parameter (e.g., $T_{14}$) is $T_{14} = 3 \pm nan$.
      - `drop_mode = drop`
         - Result: $T_{14} = nan \pm nan$
      - `drop_mode = replace`
         - Result: $T_{14} = 3 \pm 0$
   - This can significantly alter the results of large samples. For this reasons, we recommend using `drop` mode, which is safer approach.
   - This is particularly important when working with NASA tables, that have some poorly defined entries.

## NASA exoplanet archive:
Three different versions of NASA exoplanet archive Planetary System catalog are available.

1. `NASA_Exoplanet_Archive_CompositeDefault`
    - The default Composite table as extracted from NASA archive
    - Values with no errorbars are either droped or replaced with 0, based on user input as described with the `drop_mode`
2. `NASA_Exoplanet_Archive_CompositeMostPrecise`
    - Composite table with the most precise values to date.
    - This uses full table and filters the most precise values for each key. 
    - Not Implemented Yet
3. `NASA_Exoplanet_Archive_FullTable`
    - Full table as loaded from the NASA archive
    - Extremely heavy and takes a long time to load. Use the loading function when possible.
    - Not Implemented Yet

The difference between `NASA_Exoplanet_Archive_CompositeDefault` and `NASA_Exoplanet_Archive_CompositeMostPrecise` is that the former does not necessarily provide the most precise values. Using the latter is more preferable, as the values are selected based on the $1\sigma$ errorbars on the values. Furthermore, the former also occassionally puts values without errorbars, even if values with errorbar are available. This removes some planets from the sample, even if it is possible to observe.

Keep in mind the FullTable takes a long time, especially in jupyter notebooks.

In [ ]:
from PTO.database.NASA_exoplanet_archive import NASA_Exoplanet_Archive_CompositeDefault, NASA_Exoplanet_Archive_FullTable, NASA_Exoplanet_Archive_CompositeMostPrecise

# Takes very long time to load (~ 1 hour)
FullTable = NASA_Exoplanet_Archive_FullTable()
FullTable.load_API_table(force_load=True)

CompositeTable = NASA_Exoplanet_Archive_CompositeDefault()
CompositeTable.load_API_table(force_load=True)

# To be added
# CompositeMostPreciseTable = NASA_Exoplanet_Archive_CompositeDefault()
# CompositeMostPreciseTable.load_API_table(force_load=True)